In [ ]:
import pandas as pd
import numpy as np

## Map

You may recall the `map` function from the python standard library, which takes a function and a sequence as inputs, it runs the function on each of the items in the sequence and accumulates the results into a new sequence.

We have a [similar map method for pandas Series](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.map.html), but it's a little more flexible.

In [ ]:
my_list = ['cat', 'dog', '', 'rabbit']

In [ ]:
list(map(len, my_list))

In [ ]:
list(map(lambda x: x.upper(), my_list))

In [ ]:
s = pd.Series(['cat', 'dog', np.nan, 'rabbit'])
s

In [ ]:
# na_action='ignore' will ignore the NaN values, not passing them to the function
s.map(len, na_action='ignore')

In [ ]:
# instead of passing a function, you can pass a dictionary
s.map({'cat': 'kitten', 'dog': 'puppy'})

In [ ]:
# you can also pass a formatting string
s.map('I am a {}'.format)


In [ ]:
# ignore the NaN values
s.map('I am a {}'.format, na_action='ignore')

## Apply

Analogous to `map`, we can `apply` the same transformation operation to all the Series in a DataFrame. Now, we will have a function that operates on a whole Series at a time. And we will apply that function to all the columns in a DataFrame, or to all the rows.

Let's work with our powerball dataset again, so we can see how apply can be useful.

In [ ]:
powerball_df = pd.read_csv('sample_data/powerball_winners.csv')
powerball_df['Draw Date'] = pd.to_datetime(powerball_df['Draw Date'])
powerball_df.head(3)

Notice that the 'Winning Numbers' column is a string of six two-character numbers. Let's make it into six separate columns instead.

Pandas provides a [`.str.split()` method](https://pandas.pydata.org/pandas-docs/version/2.2/reference/api/pandas.Series.str.split.html) for Series objects. It's running an operation like map behind the scenes, taking each string in the series and running the python string split method on it.


In [ ]:
powerball_df['Winning Numbers'].str.split().head(3)

In [ ]:
# I can specify that I want to expand the split strings into separate columns
# Now I get a dataframe with six columns instead of a Series of lists
powerball_df['Winning Numbers'].str.split(expand=True).head(3)

In [ ]:
# So let's assign the six new columns to the dataframe
new_cols = ['Num1', 'Num2', 'Num3', 'Num4', 'Num5', 'Num6']
powerball_df[new_cols] = powerball_df['Winning Numbers'].str.split(expand=True)
powerball_df.head(3)

In [ ]:
# But the new columns are still strings; they should be numbers
powerball_df.dtypes

### Use `apply` to convert the strings to integers
So, let's use `apply` to run a string to number converter on each of those new columns. While `map` transforms each element of a Series, `apply` applies a function to each Series in a dataframe.

The function we want to apply to each column actually is a mapping operation, converting each of the elements to an integer.

In [ ]:
def convert_to_int(column):
    return pd.Series(map(lambda x: int(x), column))

powerball_df[new_cols] = powerball_df[new_cols].apply(convert_to_int)
powerball_df.dtypes

Rather than using convert_to_int, I could have used a function that's provided by pandas, called `to_numeric`. It's better to use it because it will convert to floats or integers, as needed, and has some error-handling capability, in case some of the elements can't be converted to integers.

In [ ]:
# Convert the new columns to integers
powerball_df[new_cols] = powerball_df[new_cols].apply(pd.to_numeric)
powerball_df.dtypes

In [ ]:
# specify axis=1 to indicate it's a column, not a row that we are dropping.
powerball_df.drop('Winning Numbers', axis=1).head(3)

In [ ]:
# Wait, what's going on here? The column is back!
powerball_df.head(3)

## Pandas methods generally make copies
 Most pandas methods that operate on dataframes produce new dataframes, without affecting the original. This avoids some of the confusion we had with numpy making views when we took slices, for example. Then, when we altered the slice, the original numpy array was also changed.

 In pandas, we have the opposite problem. When you are trying to make a modification to a dataframe rather than producing a new dataframe with some copied data, you have to explicitly say so. There are two ways to do that, reassigning, or specifying the `inplace` flag when invoking the operation.

In [ ]:
#reassign the dataframe to itself
powerball_df = powerball_df.drop('Winning Numbers', axis=1)
powerball_df.head(3)

In [ ]:
# or use the inplace parameter
powerball_df.drop('Num1', axis=1, inplace=True)
powerball_df.head(3)